# Arabic Audio Dataset Exploration and Preparation

## Project Overview

This notebook explores and prepares an Arabic audio dataset for the graduation project, which aims to recognize spoken Arabic letters and convert them into Braille output.

The main objectives are to:
- Explore the dataset and its class distribution.
- Inspect audio properties such as duration, sample rate, and number of channels.
- Check audio files for reading errors and potential silence.
- Create a metadata file containing audio paths and labels.
- Split the dataset into training, validation, and test sets.
- Visualize an example audio waveform.

In [1]:
from pathlib import Path

DATA_DIR = Path("../data/raw/Cleaned Audio Dataset")

print("Dataset exists:", DATA_DIR.exists())
print("Dataset path:", DATA_DIR.resolve())

Dataset exists: False
Dataset path: /data/raw/Cleaned Audio Dataset


In [2]:

from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

files_per_letter = Counter()

for folder in DATA_DIR.iterdir():   # بنشوف عندنا كام ملف في كل حرف
    if folder.is_dir():
        files_per_letter[folder.name] = sum(
            1 for file in folder.iterdir()
            if file.is_file() and file.suffix.lower() == ".wav"
        )

counts_df = pd.DataFrame(
    files_per_letter.items(),
    columns=["Letter", "Number_of_Audio_Files"]
).sort_values("Letter")

print("Number of letters:", len(counts_df))
print("Total audio files:", counts_df["Number_of_Audio_Files"].sum())

display(counts_df)


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/Cleaned Audio Dataset'

In [ ]:
  # بنرسم توزيع الداتا عشان نشوف كل حرف عنده كام عينة
plt.figure(figsize=(14, 6))

plt.bar(
    counts_df["Letter"],
    counts_df["Number_of_Audio_Files"]
)

plt.title("Number of Audio Files per Arabic Letter")
plt.xlabel("Letter")
plt.ylabel("Number of Audio Files")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#  no.of channels بنشوف مدة الصوت والـ sample rate و


import wave
from statistics import mean
from collections import Counter

durations = []
sample_rates = Counter()
channels_count = Counter()
errors = []

for file_path in DATA_DIR.rglob("*.wav"):
    try:
        with wave.open(str(file_path), "rb") as audio:
            sample_rate = audio.getframerate()
            channels = audio.getnchannels()
            frames = audio.getnframes()

            duration = frames / sample_rate

            durations.append(duration)
            sample_rates[sample_rate] += 1
            channels_count[channels] += 1

    except Exception as error:
        errors.append((file_path, str(error)))

print("Readable audio files:", len(durations))
print("Files with errors:", len(errors))

if durations:
    print(f"Shortest duration: {min(durations):.2f} seconds")
    print(f"Longest duration: {max(durations):.2f} seconds")
    print(f"Average duration: {mean(durations):.2f} seconds")

print("\nSample Rates:")
for rate, count in sorted(sample_rates.items()):
    print(f"{rate} Hz: {count} files")

print("\nChannels:")
for channels, count in sorted(channels_count.items()):
    print(f"{channels} channel(s): {count} files")


In [ ]:
# بنتأكد إن الملفات بتتقرأ ومفيش مشاكل واضحة فيها
from IPython.display import Audio, display
import random

# اختيار تسجيل عشوائي من فولدر Ain
ain_files = list((DATA_DIR / "Ain").glob("*.wav"))
sample_file = random.choice(ain_files)

print("File:", sample_file.name)
print("Label from folder:", sample_file.parent.name)

display(Audio(filename=str(sample_file)))



In [ ]:

from IPython.display import Audio, display
import random

letters_to_check = ["Alif", "Ba", "Thal"]

for letter in letters_to_check:
    files = list((DATA_DIR / letter).glob("*.wav"))
    sample_file = random.choice(files)

    print(f"Letter: {letter}")
    print(f"File: {sample_file.name}")
    display(Audio(filename=str(sample_file)))


In [ ]:

from pathlib import Path
import pandas as pd

# مكان ملفات الصوت
DATA_DIR = Path("../data/raw/Cleaned Audio Dataset")

# مكان حفظ ملف الـ metadata
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# جمع ملفات الصوت وربط كل ملف بالـ label بتاعه
records = []

for file_path in sorted(DATA_DIR.rglob("*.wav")):
    records.append({
        "file_path": file_path.relative_to("..").as_posix(),
        "label": file_path.parent.name
    })

# تحويل البيانات إلى DataFrame
metadata = pd.DataFrame(records)

# حفظ الملف
output_file = OUTPUT_DIR / "metadata.csv"
metadata.to_csv(output_file, index=False)


print("Total audio files:", len(metadata))
print("Number of labels:", metadata["label"].nunique())
print("Duplicate rows:", metadata.duplicated().sum())
print("\nFiles per label:")
print(metadata["label"].value_counts().sort_index())
print("\nFirst 5 rows:")
display(metadata.head())

print("\nSaved to:", output_file.resolve())



In [ ]:

# التأكد إن كل مسارات الملفات موجودة فعلًا
project_root = Path("..").resolve()

metadata["file_exists"] = metadata["file_path"].apply(
    lambda path: (project_root / path).exists()
)

print("Missing files:", (~metadata["file_exists"]).sum())
print("Missing labels:", metadata["label"].isna().sum())
print("Unique file paths:", metadata["file_path"].nunique())

display(metadata[["file_path", "label"]].head())


metadata.drop(columns=["file_exists"], inplace=True)


In [ ]:

output_file = OUTPUT_DIR / "metadata.csv"
metadata.to_csv(output_file, index=False)

print("Metadata saved successfully!")
print("Total records:", len(metadata))
print("Saved at:", output_file.resolve())


In [ ]:

from sklearn.model_selection import train_test_split

# 1. فصل بيانات التدريب عن باقي البيانات
train_df, temp_df = train_test_split(
    metadata,
    test_size=0.30,
    random_state=42,
    stratify=metadata["label"]
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nNumber of labels in each set:")
print("Train:", train_df["label"].nunique())
print("Validation:", val_df["label"].nunique())
print("Test:", test_df["label"].nunique())


In [ ]:

# حفظ تقسيمات الداتا
# بنحفظ كل مجموعة في ملف CSV لوحدها
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

print("All splits saved successfully!")
print("Train:", (OUTPUT_DIR / "train.csv").resolve())
print("Validation:", (OUTPUT_DIR / "validation.csv").resolve())
print("Test:", (OUTPUT_DIR / "test.csv").resolve())


In [ ]:

import numpy as np
import wave
import pandas as pd

silence_candidates = []
rms_values = []

for _, row in metadata.iterrows():
    file_path = project_root / row["file_path"]

    with wave.open(str(file_path), "rb") as audio_file:
        frames = audio_file.readframes(audio_file.getnframes())
        sample_width = audio_file.getsampwidth()

    # تحويل بيانات WAV إلى أرقام
    if sample_width == 2:
        audio = np.frombuffer(frames, dtype="<i2").astype(np.float64)
        audio /= 32768.0
    elif sample_width == 4:
        audio = np.frombuffer(frames, dtype="<i4").astype(np.float64)
        audio /= 2147483648.0
    elif sample_width == 1:
        audio = np.frombuffer(frames, dtype=np.uint8).astype(np.float64)
        audio = (audio - 128) / 128.0
    else:
        raise ValueError(f"Unsupported sample width: {sample_width}")

    rms = np.sqrt(np.mean(audio ** 2))
    rms_values.append(rms)

    if rms < 0.001:
        silence_candidates.append({
            "file_path": row["file_path"],
            "label": row["label"],
            "rms": rms
        })

print("Total files checked:", len(rms_values))
print("Very low-volume candidates:", len(silence_candidates))

display(pd.DataFrame(silence_candidates).head(10))



In [ ]:
# بنرسم شكل الإشارة الصوتية لعينة من الداتا
import matplotlib.pyplot as plt
import numpy as np
import wave


sample_file = next((DATA_DIR / "Ain").glob("*.wav"))

with wave.open(str(sample_file), "rb") as audio_file:
    sample_rate = audio_file.getframerate()
    n_frames = audio_file.getnframes()
    sample_width = audio_file.getsampwidth()
    frames = audio_file.readframes(n_frames)

if sample_width == 2:
    audio = np.frombuffer(frames, dtype="<i2").astype(np.float32) / 32768.0
elif sample_width == 4:
    audio = np.frombuffer(frames, dtype="<i4").astype(np.float32) / 2147483648.0
elif sample_width == 1:
    audio = (np.frombuffer(frames, dtype=np.uint8).astype(np.float32) - 128) / 128.0
else:
    raise ValueError(f"Unsupported sample width: {sample_width}")

time = np.arange(len(audio)) / sample_rate

plt.figure(figsize=(12, 4))
plt.plot(time, audio)
plt.title(f"Waveform - {sample_file.name}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(0, len(audio) / sample_rate)
plt.grid(True)
plt.show()

print("Sample rate:", sample_rate, "Hz")
print("Duration:", len(audio) / sample_rate, "seconds")
print("Maximum amplitude:", np.max(np.abs(audio)))


In [ ]:
%pip install librosa

In [ ]:
import librosa
print("Librosa version:", librosa.__version__)

el code da test lel function ba3d ma katbnaha fe data_prep.py

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from data_prep import load_audio

test_file = next((DATA_DIR / "Ain").glob("*.wav"))

audio, sample_rate = load_audio(str(test_file))

print("File:", test_file.name)
print("Audio shape:", audio.shape)
print("Sample rate:", sample_rate)
print("Duration:", len(audio) / sample_rate, "seconds")
print("Audio dtype:", audio.dtype)

In [ ]:
from data_prep import get_splits

train_df, val_df, test_df = get_splits()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Train labels:", train_df["label"].nunique())

In [ ]:
import importlib
import data_prep

importlib.reload(data_prep)

print(hasattr(data_prep, "get_splits"))

In [ ]:
from pathlib import Path

processed_dir = Path("../data/processed")

for filename in ["metadata.csv", "train.csv", "val.csv", "test.csv"]:
    file_path = processed_dir / filename
    print(filename, "Exists:", file_path.exists())

In [ ]:
total = len(train_df) + len(val_df) + len(test_df)

print(f"Train: {len(train_df) / total:.1%}")
print(f"Validation: {len(val_df) / total:.1%}")
print(f"Test: {len(test_df) / total:.1%}")
print(f"Total files: {total}")

In [ ]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print("Number of classes:", df["label"].nunique())
    print("Missing values:", df[["file_path", "label"]].isna().sum().sum())
    print("Duplicate file paths:", df["file_path"].duplicated().sum())

In [ ]:
train_paths = set(train_df["file_path"])
val_paths = set(val_df["file_path"])
test_paths = set(test_df["file_path"])

print("Train ∩ Validation:", len(train_paths & val_paths))
print("Train ∩ Test:", len(train_paths & test_paths))
print("Validation ∩ Test:", len(val_paths & test_paths))

print("Total unique files:", len(train_paths | val_paths | test_paths))

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from data_prep import get_splits
import pandas as pd

#extracting the number of each letter in the dataset

import json  #to store the table in a json file

sys.path.insert(0, str(Path("..").resolve()))

train_df, val_df, test_df = get_splits()  # load the saved train/val/test splits

 #extracting the number of each letter in the dataset

for df in (train_df, val_df, test_df):
    df["label_id"] = df["file_path"].apply(lambda p: int(Path(p).name.split("_")[0]))

# Collect the unique (letter name, number) pairs, sorted by number
all_df = pd.concat([train_df, val_df, test_df])
pairs = all_df[["label", "label_id"]].drop_duplicates().sort_values("label_id")

# Sanity check

assert len(pairs) == pairs["label_id"].nunique() == pairs["label"].nunique() == 28

display(pairs)


In [ ]:
#make a dic and store it in a json file

# Letter name -> number (int() is needed because json cannot save numpy ints)
label2id = {l: int(i) for l, i in zip(pairs["label"], pairs["label_id"])}

# Number -> letter name (used to turn model predictions back into letters)
id2label = {i: l for l, i in label2id.items()}

# Create the output folder if it does not exist, then save label2id as a JSON file
Path("../data/processed").mkdir(parents=True, exist_ok=True)
with open("../data/processed/label2id.json", "w") as f:
    json.dump(label2id, f, indent=2)

print(label2id)

In [ ]:
# Save each split as a CSV file that now includes the label_id column
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert df["label_id"].notna().all(), f"{name}: some rows have no label_id"
    df.to_csv(f"../data/processed/{name}.csv", index=False)

In [ ]:
# Check each split: number of rows, number of classes, and id range
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, "| rows:", len(df), "| classes:", df["label_id"].nunique(),
          "| min id:", df["label_id"].min(), "| max id:", df["label_id"].max())

# Round-trip test: letter -> number -> letter should return the same letter
sample = train_df.iloc[0]
print(sample["label"], "->", sample["label_id"], "->", id2label[sample["label_id"]])